# WIS2 pygeoAPI — Query Notebook

This notebook demonstrates how to query the IMOS WIS2 node OGC API (powered by **pygeoapi**).

**Base URL:** `https://wis2box.production.aodn.org.au/oapi`

**Collections available:**

| Collection | Contents |
|---|---|
| `discovery-metadata` | WCMP2 dataset records |
| `stations` | WIGOS-registered wave buoy stations |
| `messages` | WIS Notification Messages (with BUFR download links) |
| `urn:wmo:md:au-imos:wave-buoys` | Decoded observations (GeoJSON) |

Reference: [`docs/pygeoAPI.md`](../../docs/pygeoAPI.md)


## 0. Setup

In [ ]:
import requests
import json
import pandas as pd

BASE_URL = "https://wis2box.production.aodn.org.au/oapi"

def get(path, **params):
    """GET helper — always requests JSON, prints status."""
    params.setdefault("f", "json")
    r = requests.get(f"{BASE_URL}{path}", params=params)
    r.raise_for_status()
    return r.json()


## 1. API Landing Page

The landing page lists all available links including the OpenAPI spec and collections endpoint.


In [ ]:
landing = get("")
for link in landing["links"]:
    print(f"{link['rel']:20s}  {link['href']}")


## 2. List Collections

Each collection has an `id`, a human-readable `title`, and spatial/temporal extent information.


In [ ]:
collections = get("/collections")
for c in collections["collections"]:
    print(f"{c['id']:45s}  {c['title']}")


## 3. Discovery Metadata

The `discovery-metadata` collection contains WCMP2 records describing published datasets.


In [ ]:
records = get("/collections/discovery-metadata/items")
print(f"Total records: {records['numberMatched']}")
for f in records["features"]:
    print(f"  {f['id']}")
    print(f"    title: {f['properties'].get('title', '')}")


In [ ]:
# Fetch the wave-buoys record directly
record = get("/collections/discovery-metadata/items/urn:wmo:md:au-imos:wave-buoys")
print(json.dumps(record["properties"], indent=2, default=str))


## 4. Stations

The `stations` collection lists all 23 WIGOS-registered wave buoy stations.

**Queryable fields:** `wigos_station_identifier`, `name`, `facility_type`,
`territory_name`, `wmo_region`, `status`, `topic`


In [ ]:
# All stations as a DataFrame
result = get("/collections/stations/items", limit=100)
print(f"Total stations: {result['numberMatched']}")

stations_df = pd.json_normalize([f["properties"] for f in result["features"]])
stations_df = stations_df[["name", "wigos_station_identifier", "status",
                            "facility_type", "wmo_region", "territory_name"]]
stations_df


In [ ]:
# Fetch a single station by WIGOS ID
wigos_id = "0-22000-0-7811080"
station = get(f"/collections/stations/items/{wigos_id}")
print(json.dumps(station["properties"], indent=2))


In [ ]:
# Stations within a bounding box (south-east Australia)
result = get("/collections/stations/items", bbox="140,-40,155,-30", limit=50)
print(f"Stations in SE Australia: {result['numberMatched']}")
for f in result["features"]:
    p = f["properties"]
    lon, lat = f["geometry"]["coordinates"][:2]
    print(f"  {p['name']:25s}  {wigos_id:25s}  ({lat:.4f}, {lon:.4f})")


## 5. Notifications (`messages`)

The `messages` collection contains one **WIS Notification Message (WNM)** per published BUFR file.
Each feature includes:
- `properties.data_id` — unique identifier
- `properties.datetime` — observation time
- `properties.pubtime` — publication time
- `properties.wigos_station_identifier`
- `links[].href` (rel=`canonical`) — direct BUFR4 download URL

**Queryable fields:** `wigos_station_identifier`, `data_id`, `datetime`, `pubtime`, `metadata_id`


In [ ]:
# Check queryable fields
queryables = get("/collections/messages/queryables")
for name, info in queryables["properties"].items():
    print(f"  {name:35s}  {info.get('type', '')}")


In [ ]:
# Latest 10 notifications for Apollo Bay
wigos_id = "0-22000-0-7811080"
result = get("/collections/messages/items",
             wigos_station_identifier=wigos_id, limit=10)

print(f"Total notifications for {wigos_id}: {result['numberMatched']}")

rows = []
for f in result["features"]:
    p = f["properties"]
    bufr_url = next((l["href"] for l in f["links"] if l["rel"] == "canonical"), None)
    rows.append({
        "datetime":   p["datetime"],
        "pubtime":    p["pubtime"],
        "data_id":    p["data_id"],
        "bufr_url":   bufr_url,
    })

pd.DataFrame(rows)


In [ ]:
# Filter by station + datetime range
result = get("/collections/messages/items",
             wigos_station_identifier="0-22000-0-7811080",
             datetime="2025-11-01/2025-11-30",
             limit=10)

print(f"Matched: {result['numberMatched']}")
for f in result["features"]:
    p = f["properties"]
    print(f"  {p['datetime']}  {p['data_id']}")


In [ ]:
# Free-text search using q= (searches all text fields including data_id)
result = get("/collections/messages/items",
             q="WIGOS_0-22000-0-7811080",
             limit=5)
print(f"Matched: {result['numberMatched']}")
for f in result["features"]:
    print(f"  {f['properties']['data_id']}")


In [ ]:
# Spatial filter — all notifications within a bounding box
result = get("/collections/messages/items",
             bbox="143.0,-39.0,144.5,-38.0",
             limit=5)
print(f"Notifications in bbox: {result['numberMatched']}")
for f in result["features"]:
    p = f["properties"]
    print(f"  {p['datetime']}  {p['wigos_station_identifier']}  {p['data_id'][:60]}")


## 6. Pagination

The API is offset-based. Use `numberMatched` to know the total and increment `offset` by `limit`.


In [ ]:
def fetch_all_messages(wigos_id, datetime_range, page_size=100):
    """Fetch all notifications for a station in a date range."""
    params = {
        "wigos_station_identifier": wigos_id,
        "datetime": datetime_range,
        "limit": page_size,
        "offset": 0,
    }
    features = []
    while True:
        page = get("/collections/messages/items", **params)
        features.extend(page["features"])
        total = page["numberMatched"]
        print(f"  Fetched {len(features)}/{total}")
        if len(features) >= total:
            break
        params["offset"] += page_size
    return features

features = fetch_all_messages("0-22000-0-7811080", "2025-11-01/2025-11-30")
print(f"\nTotal fetched: {len(features)}")


## 7. Download BUFR Files

Each notification's `links` array contains a `canonical` link to the BUFR4 file.


In [ ]:
# Fetch the first notification for Apollo Bay and download its BUFR file
result = get("/collections/messages/items",
             wigos_station_identifier="0-22000-0-7811080",
             limit=1)

feature = result["features"][0]
props   = feature["properties"]
bufr_url = next(l["href"] for l in feature["links"] if l["rel"] == "canonical")

print(f"data_id  : {props['data_id']}")
print(f"datetime : {props['datetime']}")
print(f"BUFR URL : {bufr_url}")

bufr_bytes = requests.get(bufr_url).content
print(f"\nDownloaded {len(bufr_bytes)} bytes")
print(f"BUFR magic bytes: {bufr_bytes[:4]}")   # should be b'BUFR'


In [ ]:
# Save to disk
import pathlib

outdir = pathlib.Path("data/bufr")
outdir.mkdir(parents=True, exist_ok=True)

filename = outdir / bufr_url.split("/")[-1]
filename.write_bytes(bufr_bytes)
print(f"Saved: {filename}")


## Summary

| Task | Pattern |
|---|---|
| List collections | `GET /oapi/collections` |
| Get discovery record | `GET /oapi/collections/discovery-metadata/items/{id}` |
| All stations | `GET /oapi/collections/stations/items` |
| One station | `GET /oapi/collections/stations/items/{wigos_id}` |
| Notifications for station | `?wigos_station_identifier=0-22000-0-7811080` |
| Filter by date | `?datetime=2026-05-01/2026-05-07` |
| Spatial filter | `?bbox=minLon,minLat,maxLon,maxLat` |
| Free-text search | `?q=WIGOS_0-22000-0-7811080` |
| Paginate | `?limit=100&offset=N` |
| Download BUFR | Follow `links[rel=canonical].href` |

See [`docs/pygeoAPI.md`](../../docs/pygeoAPI.md) for the full reference.
